In [1]:
# Check GPU
!nvidia-smi

Mon Feb  2 07:22:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P0             27W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip uninstall -y transformers peft accelerate

Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: peft 0.17.1
Uninstalling peft-0.17.1:
  Successfully uninstalled peft-0.17.1
Found existing installation: accelerate 1.11.0
Uninstalling accelerate-1.11.0:
  Successfully uninstalled accelerate-1.11.0


In [3]:
# Install dependencies
%pip install -q transformers peft accelerate evaluate datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 93.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 32.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 31.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [4]:
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_from_disk
import torch
from pathlib import Path
import logging
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from peft import PeftModel
import shutil
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB


In [ ]:
# CONFIG 

EPOCHS = 3                    
LEARNING_RATE = 2e-5           
BATCH_SIZE = 8
MAX_LENGTH = 384
DOC_STRIDE = 128
WEIGHT_DECAY = 0.05            

STAGE1_CHECKPOINT = "/kaggle/input/xlm-roberta-stage-1-best/stage1_best"
DATASET_PATH = "/kaggle/input/viquad-augmented-context-aware-55k/viquad_augmented"  

OUTPUT_DIR = "/kaggle/working/stage2_output"
CHECKPOINT_DIR = "/kaggle/working/stage2_checkpoints"
FINAL_MODEL_DIR = "/kaggle/working/stage2_best"

print("Training Configuration (ANTI-OVERFITTING):")
print(f"  Dataset: Augmented ViQuAD (55K samples)")
print(f"  Base model: XLM-RoBERTa (from Stage 1)")
print(f"  Epochs: {EPOCHS} ← REDUCED (overfits after ~2000 steps)")
print(f"  Learning rate: {LEARNING_RATE} ← DECREASED for stability")
print(f"  Weight decay: {WEIGHT_DECAY} ← INCREASED for regularization")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Effective batch: {BATCH_SIZE * 4} (with gradient accumulation)")
print(f"\n  Strategy: Stop training before overfitting kicks in")
print(f"  Expected: Better generalization → Higher F1 on test set")

Training Configuration:
  Dataset: Augmented ViQuAD (55K samples)
  Base model: XLM-RoBERTa (from Stage 1)
  Epochs: 5
  Learning rate: 3e-05
  Batch size: 8


## Load Stage 1 Checkpoint

In [6]:
# Copy sang working directory
TMP_CHECKPOINT = "/kaggle/working/stage1_best"
os.makedirs(TMP_CHECKPOINT, exist_ok=True)

for file in os.listdir(STAGE1_CHECKPOINT):
    src = os.path.join(STAGE1_CHECKPOINT, file)
    dst = os.path.join(TMP_CHECKPOINT, file)
    if os.path.isfile(src):
        shutil.copy(src, dst)

# Load tokenizer từ checkpoint đã copy
tokenizer = AutoTokenizer.from_pretrained(TMP_CHECKPOINT)

# Load base model
base_model = AutoModelForQuestionAnswering.from_pretrained("xlm-roberta-base")

# Load PEFT adapter
model = PeftModel.from_pretrained(base_model, TMP_CHECKPOINT)

# Merge adapter vào model
model = model.merge_and_unload()

for param in model.parameters():
    param.requires_grad = True
print(f"Model parameters: {model.num_parameters():,}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base/xet-read-token/e73636d4f797dec63c3081bb6ed5c7b0bb3f2089 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForQuestionAnswering LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
qa_outputs.bias             | MISSING    | 
qa_outputs.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 277,454,594


## Load ViQuAD Dataset

In [7]:
print(f"Loading ViQuAD Augmented from {DATASET_PATH}")
viquad = load_from_disk(DATASET_PATH)

print("\nDataset Statistics:")
print(f"  Train:      {len(viquad['train']):6,} samples  (augmented 2x)")
print(f"  Validation: {len(viquad['validation']):6,} samples  (unchanged)")
if 'test' in viquad:
    print(f"  Test:       {len(viquad['test']):6,} samples  (unchanged)")

# Check augmentation rate
train_data = list(viquad['train'])
original_count = len([ex for ex in train_data if '_aug' not in ex['id']])
augmented_count = len([ex for ex in train_data if '_aug' in ex['id']])

print(f"\nAugmentation Details:")
print(f"  Original examples: {original_count:,}")
print(f"  Augmented examples: {augmented_count:,}")
print(f"  Total: {len(train_data):,}")
print(f"  Increase: +{(len(train_data)/original_count - 1)*100:.1f}%")

# Show sample augmented example
aug_sample = next((ex for ex in train_data if '_aug1' in ex['id']), None)
if aug_sample:
    orig_id = aug_sample['id'].split('_aug')[0]
    orig_sample = next((ex for ex in train_data if ex['id'] == orig_id), None)
    if orig_sample:
        print(f"\nSample Augmentation:")
        print(f"  Original:  {orig_sample['question'][:80]}...")
        print(f"  Augmented: {aug_sample['question'][:80]}...")

Loading ViQuAD Augmented from /kaggle/input/viquad-augmented-context-aware-55k/viquad_augmented

Dataset Statistics:
  Train:      54,892 samples  (augmented 2x)
  Validation:  3,814 samples  (unchanged)
  Test:        7,301 samples  (unchanged)

Augmentation Details:
  Original examples: 28,454
  Augmented examples: 26,438
  Total: 54,892
  Increase: +92.9%

Sample Augmentation:
  Original:  Tên gọi nào được Phạm Văn Đồng sử dụng khi làm Phó chủ nhiệm cơ quan Biện sự xứ ...
  Augmented: Tên gọi nào được Phạm Văn Đồng sử dụng khi tiến hành Phó chủ nhiệm cơ quan Biện ...


## Prepare Data

In [8]:
def prepare_train_features(examples):
    tokenized = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    
    tokenized["start_positions"] = []
    tokenized["end_positions"] = []
    
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        
        # Nếu không có answer, đặt vị trí là CLS token
        if len(answers["answer_start"]) == 0:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
            continue
        
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        
        # Lấy sequence_ids để xác định context
        sequence_ids = tokenized.sequence_ids(i)
        
        # Tìm vị trí bắt đầu và kết thúc của context
        context_start = sequence_ids.index(1) if 1 in sequence_ids else 0
        context_end = len(sequence_ids) - 1 - sequence_ids[::-1].index(1) if 1 in sequence_ids else len(sequence_ids)
        
        # Tìm token start index
        token_start_index = context_start
        while token_start_index <= context_end and offsets[token_start_index][0] <= start_char:
            token_start_index += 1
        token_start_index -= 1
        
        # Tìm token end index
        token_end_index = context_end
        while token_end_index >= context_start and offsets[token_end_index][1] >= end_char:
            token_end_index -= 1
        token_end_index += 1
        
        if (token_start_index < context_start or 
            token_end_index > context_end or
            token_start_index >= len(offsets) or
            token_end_index >= len(offsets) or
            token_start_index < 0 or
            token_end_index < 0):
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        elif not (offsets[token_start_index][0] <= start_char and 
                  offsets[token_end_index][1] >= end_char):
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        else:
            tokenized["start_positions"].append(token_start_index)
            tokenized["end_positions"].append(token_end_index)
    
    return tokenized

print("Tokenizing datasets")
train_dataset = viquad['train'].map(
    prepare_train_features,
    batched=True,
    remove_columns=viquad['train'].column_names,
    desc="Tokenizing train",
    keep_in_memory=True  # tránh lỗi read-only
)

val_dataset = viquad['validation'].map(
    prepare_train_features,
    batched=True,
    remove_columns=viquad['validation'].column_names,
    desc="Tokenizing validation",
    keep_in_memory=True #tránh lỗi read-only
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

Tokenizing datasets


Tokenizing train:   0%|          | 0/54892 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/3814 [00:00<?, ? examples/s]

Train dataset: 58664 samples
Val dataset: 3936 samples


## Training

In [ ]:
# ANTI-OVERFITTING Training Arguments
# Model overfits after step ~2000, so we optimize for early stopping

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    
    # More frequent evaluation to catch best model before overfitting
    eval_strategy="steps",  
    eval_steps=250,                    # 500 → 250 (monitor more closely)
    save_strategy="steps",
    save_steps=250,                    # 500 → 250 (save more checkpoints)
    save_total_limit=5,                # Keep 5 best checkpoints
    
    # Learning settings
    learning_rate=LEARNING_RATE,       # 2e-5 (stable)
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,           # 3 epochs (stop before overfit)
    
    # Regularization - INCREASED to prevent overfitting
    weight_decay=WEIGHT_DECAY,         # 0.05 (stronger regularization)
    warmup_ratio=0.1,                  # Warm up 10% of training
    max_grad_norm=1.0,                 # Gradient clipping
    
    # Learning rate scheduler - cosine decay to reduce LR over time
    lr_scheduler_type="cosine",        # Helps with convergence
    
    # Performance
    fp16=True,
    gradient_accumulation_steps=4,
    dataloader_num_workers=2,
    
    # Monitoring
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Misc
    report_to="none",
    push_to_hub=False
)

print("Training Strategy:")
print(f"  Total steps: ~{(len(train_dataset) // (BATCH_SIZE * 4)) * EPOCHS}")
print(f"  Eval every: 250 steps (catch best model early)")
print(f"  Best model expected around step 1500-2000")
print(f"  Early stopping will prevent overfit after step 2000")
print(f"  Stronger regularization (weight_decay=0.05) reduces overfit")
print(f"  Cosine LR decay helps smooth convergence")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)  # Stop if val_loss doesn't improve for 3 evals
    ]
)

print("\n" + "="*60)
print("Starting Stage 2 Training (ANTI-OVERFITTING)")
print("="*60)
print("Monitoring: Watch for val_loss plateau around step 2000")
print("Expected: Training will auto-stop before severe overfit")
print("="*60 + "\n")

trainer.train()

Starting Stage 2 Training (VI Fine-tune)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss,Validation Loss
500,1.839029,1.804699
1000,1.469028,1.594808
1500,1.325648,1.544124
2000,1.085410,1.533329
2500,0.989394,1.548784
3000,0.932883,1.569986
3500,0.913485,1.620369


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3500, training_loss=1.3373168324061802, metrics={'train_runtime': 4634.5571, 'train_samples_per_second': 63.29, 'train_steps_per_second': 1.979, 'total_flos': 2.194422422420275e+16, 'train_loss': 1.3373168324061802, 'epoch': 1.9087685803900176})

## Save Final Model

In [11]:
print(f"Saving best model to {FINAL_MODEL_DIR}")
print("Note: Trainer has already loaded the best checkpoint (lowest eval_loss)")
print(f"Best model will be saved to: {FINAL_MODEL_DIR}")

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Best model saved to: {FINAL_MODEL_DIR}")
print("Stage 2 completed!")


Saving best model to /kaggle/working/stage2_best
Note: Trainer has already loaded the best checkpoint (lowest eval_loss)
Best model will be saved to: /kaggle/working/stage2_best


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to: /kaggle/working/stage2_best
Stage 2 completed!


In [12]:
# Check output files
!ls -lh /kaggle/working/stage2_best/

total 1.1G
-rw-r--r-- 1 root root  757 Feb  2 08:56 config.json
-rw-r--r-- 1 root root 1.1G Feb  2 08:56 model.safetensors
-rw-r--r-- 1 root root  353 Feb  2 08:56 tokenizer_config.json
-rw-r--r-- 1 root root  16M Feb  2 08:56 tokenizer.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Quick Evaluation (Optional)

In [13]:
test_examples = viquad['validation'].select(range(10))

for i, example in enumerate(test_examples):
    print(f"Example {i+1}")
    print(f"Question: {example['question']}")
    print(f"Context: {example['context'][:200]}...")
    
    # Check if answer exists
    if len(example['answers']['text']) > 0:
        print(f"Ground Truth: {example['answers']['text'][0]}")
    else:
        print(f"Ground Truth: [NO ANSWER]")
    
    # Predict
    inputs = tokenizer(
        example['question'],
        example['context'],
        return_tensors="pt",
        truncation="only_second",
        max_length=384,
        padding=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits)
    
    answer_tokens = inputs["input_ids"][0][start_idx:end_idx + 1]
    prediction = tokenizer.decode(answer_tokens, skip_special_tokens=True)
    
    print(f"Prediction: {prediction}")

Example 1
Question: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
Context: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ 10, Paris đã là một trong những thành phố chính của Pháp cù...
Ground Truth: trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí
Prediction: 
Example 2
Question: Vị trí địa lý của Pháp có gì đặc biệt?
Context: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ 10, Paris đã là một trong những thành phố chính của Pháp cù...
Ground Truth: [NO ANSWER]
Prediction: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có
Example 3
Question: Kinh tế xung quanh kinh đô ánh sáng mạnh về gì?
Context: Paris nằm ở điểm gặp nhau của các hành trình th